# 02 — Raw Ingestion

Read the source CSV with Spark and write it to the Raw layer in MinIO.

## Spark Session

This session is configured to access MinIO through the S3A connector.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("lakehouse-raw-ingestion") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://lakehouse-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark

In [ ]:
raw_path = "s3a://lakehouse/raw/online_retail/"
source_path = "/home/jovyan/data/online_retail.csv"

df_raw = spark.read.csv(
    source_path,
    header=True,
    inferSchema=True
)

df_raw.show(5)

In [ ]:
df_raw.printSchema()
df_raw.count()

In [ ]:
df_raw.write.mode("overwrite").parquet(raw_path)

In [ ]:
df_raw_check = spark.read.parquet(raw_path)
df_raw_check.show(5)

In [ ]:
df_raw_check.count()

## Result

The original dataset is now available in the Raw layer of the lakehouse.